In [32]:
# Setup
from pathlib import Path

import pandas as pd
import numpy as np

ica = pd.read_parquet("../data/public_products_2026-10-20.parquet")
coop = pd.read_parquet("../data/coop_products_2026-10-20.parquet")

NAME_COL = "name"

In [33]:
# Some basic data exploration
print("shapes (ica, coop): ", ica.shape, coop.shape)

#print(ica.sample(15)[[NAME_COL]])

na = pd.DataFrame({
  "ica": ica.isna().mean(),
  "coop": coop.isna().mean(),
})

print(na.sort_values("coop", ascending=False))

shapes (ica, coop):  (57016, 16) (32602, 16)
                          ica      coop
is_alcohol           0.000000  1.000000
ingredients          0.431809  0.422673
product_information  0.216851  0.376879
country_of_origin    0.888277  0.243727
brand                0.041550  0.044138
unit                 0.000000  0.016379
pack_size            0.273309  0.003742
avg_price            0.000000  0.000000
first_seen_at        0.000000  0.000000
last_seen_at         0.000000  0.000000
name                 0.000000  0.000000
product_id           0.000000  0.000000
product_image_url    0.000000  0.000000
product_key          0.000000  0.000000
product_type         0.000000  0.000000
search_vector        0.000000  0.000000


In [ ]:
# Text normalization
import re

ACCENTS = str.maketrans("éèêëáàâüûúíìïóòôçñ", "eeeeaaauuuiiiooocn")

def norm_text(s):
    # Lowercase, strip accents (keeping å/ä/ö), keep only letters, digits, decimals and %.
    s = s.fillna("").str.lower().str.translate(ACCENTS)
    s = s.str.replace(r"['’´`]", "", regex=True)                # L’Oréal -> loreal
    s = s.str.replace(r"(\d),(\d)", r"\1.\2", regex=True)       # 2,8 -> 2.8
    s = s.str.replace(r"(\d)\s+%", r"\1%", regex=True)           # 38 % -> 38%
    s = s.str.replace(r"[^a-z0-9åäö.%]+", " ", regex=True)
    s = s.str.replace(r"(?<!\d)\.|\.(?!\d)", " ", regex=True)    # keep dots only inside numbers
    return s.str.split().str.join(" ")

# Same brand written differently in the two stores (normalized form -> canonical form)
BRAND_ALIASES = {"loreal": "lorealparis"}

def norm_brand(s):
    # Like norm_text but without spaces, so 'Eget Bageri' and 'egetbageri' become the same.
    s = norm_text(s).str.replace(r" ab$", "", regex=True)  # drop company suffix: 'Torfolk Gård AB'
    return s.str.replace(" ", "", regex=False).replace(BRAND_ALIASES)

In [ ]:
# Pack size parsing: everything becomes (value, unit) with unit in g / ml / st
# (no meters: '8m' on baby food means 8 months, which is far more common than e.g. foil length)
SIZE_RE = re.compile(
    r"(?:\bca\.?\s*)?(?:(\d+)\s*x\s*)?(\d+(?:[.,]\d+)?)\s*-?\s*"
    r"(kg|kilo|g|gr|gram|l|lit|liter|dl|cl|ml|st|styck|p|pack|port|tabs|per frp)\b",
    re.IGNORECASE,
)
UNITS = {  # unit -> (base unit, multiplier)
    "kg": ("g", 1000), "kilo": ("g", 1000), "g": ("g", 1), "gr": ("g", 1), "gram": ("g", 1),
    "l": ("ml", 1000), "lit": ("ml", 1000), "liter": ("ml", 1000),
    "dl": ("ml", 100), "cl": ("ml", 10), "ml": ("ml", 1),
    "st": ("st", 1), "styck": ("st", 1), "p": ("st", 1), "pack": ("st", 1),
    "port": ("st", 1), "tabs": ("st", 1), "per frp": ("st", 1),
}

def parse_size(text):
    """'0.5L' -> (500.0, 'ml'), '6x33cl' -> (1980.0, 'ml'), '44ST' -> (44.0, 'st').
    Prefers weight/volume over a count, so '2-pack 130g' -> (130.0, 'g')."""
    if not isinstance(text, str):
        return (np.nan, None)
    sizes = []
    for count, value, unit in SIZE_RE.findall(text):
        base, mult = UNITS[unit.lower()]
        value = float(value.replace(",", ".")) * mult * (int(count) if count else 1)
        sizes.append((round(value, 3), base))
    if not sizes:
        return (np.nan, None)
    return next((s for s in sizes if s[1] in ("g", "ml")), sizes[0])

for t in ["0.5L", "6x33cl", "Chokladbiskvi 2-pack 130g", "400g - 1000000g", "Ca 750 g", "44Tabs"]:
    print(f"{t!r:30} -> {parse_size(t)}")

In [ ]:
# Exclusion rules: products that can't have an identical match in the other store.
# Matched on the normalized brand, so 'ICA Maxi Borås' and 'icamaxiborås' are both caught.
# Kept narrow on purpose, since real brands like 'Dahls Bageri' and 'Arla Köket' exist in both stores.
PRIVATE_LABEL_RE = r"^(?:ica|coop|storacoop|änglamark|xtra|mywear)"
# In-store products ('Eget bageri', 'Maxis egna', 'Butiksköket', 'Vårt kök', ...)
IN_STORE_RE = r"^(?:eget|egen|egna|maxibageri|butik|vårt|våreg|vårdeli|hemlagat|delikatessen|styckadibutik)|egna|egen$"

def exclusion_reason(df):
    reason = pd.Series(None, index=df.index, dtype="object")
    reason[df["brand_norm"].str.contains(IN_STORE_RE)] = "in_store"
    reason[df["brand_norm"].str.contains(PRIVATE_LABEL_RE)] = "private_label"
    reason[df["brand_norm"] == ""] = "no_brand"
    return reason

In [37]:
# Normalize both stores into the same slim format
def brand_pattern(brand_norm):
    # Regex for the brand inside a name, ignoring spaces: 'lorealparis' matches 'loreal paris'.
    # The trailing \w* also catches the rest of a truncated brand ('...cheesecom' -> 'cheese company').
    return r"\b" + r"\s*".join(map(re.escape, brand_norm)) + r"\w*"

def normalize(df, store):
    out = pd.DataFrame({
        "store": store,
        "product_key": df["product_key"],
        "name": df["name"],
        "brand": df["brand"],
        "price": df["avg_price"],
        "image_url": df["product_image_url"],
    })
    out["brand_norm"] = norm_brand(df["brand"])

    # Size from the name if it has one (more reliable), otherwise from pack_size
    from_name = df["name"].map(parse_size)
    from_pack = df["pack_size"].map(parse_size)
    sizes = [n if n[1] else p for n, p in zip(from_name, from_pack)]
    out["size_value"] = [s[0] for s in sizes]
    out["size_unit"] = [s[1] for s in sizes]

    # Remove size and brand from the name so names look the same in both stores
    # (ICA often puts both in the name, Coop usually doesn't)
    name = norm_text(df["name"].fillna("").str.replace(SIZE_RE, " ", regex=True))
    out["name_norm"] = [
        " ".join(re.sub(brand_pattern(b), " ", n).split()) if b else n
        for n, b in zip(name, out["brand_norm"])
    ]

    out["excluded_reason"] = exclusion_reason(out)
    return out

products = pd.concat([normalize(ica, "ica"), normalize(coop, "coop")], ignore_index=True)
candidates = products[products["excluded_reason"].isna()]
print(candidates["store"].value_counts())

store
ica     41066
coop    26827
Name: count, dtype: int64


In [40]:
# This checks what got excluded, how often size is missing, with some random picked out examples
print(products.groupby("store")["excluded_reason"].value_counts(dropna=False).unstack(0))
print("\nshare missing size:\n", products.groupby("store")["size_unit"].apply(lambda s: s.isna().mean()))

products[products["excluded_reason"].isna()].sample(15)[
    ["store", "name", "brand_norm", "name_norm", "size_value", "size_unit"]
]

store             coop    ica
excluded_reason              
in_store             2   2584
no_brand          1442   2400
private_label     4331  10966
NaN              26827  41066

share missing size:
 store
coop    0.038341
ica     0.180651
Name: size_unit, dtype: float64


,store,name,brand_norm,name_norm,size_value,size_unit
76761,coop,Termosmugg Rosa,sensokitchen,termosmugg rosa,1.0,st
38680,ica,Svartvinbärsglögg Alkoholfri Ekologisk 650cl K...,tillmans,svartvinbärsglögg alkoholfri ekologisk krav,6500.0,ml
85101,coop,Bläckfiskringar Opanerade,feldts,bläckfiskringar opanerade,400.0,g
82009,coop,Ägg 12-pack L Frigående,ulfstorpsgårdsägg,ägg l frigående,12.0,st
26664,ica,Karréskivor benfri ca 350g,skövdeslakteri,karreskivor benfri,350.0,g
5854,ica,Vinicius Jr Äger,mugfordgreen,vinicius jr äger,NaN,NaN
45937,ica,Coconut mjölk chocolate cookie 30 g Dig,dig,coconut mjölk chocolate cookie,30.0,g
32572,ica,Surskalle Gelatinfri 190g Fairtrade Bubs,bubs,surskalle gelatinfri fairtrade,190.0,g
50957,ica,Ägg M 15-pack Frigående Uggelsta Ägg,uggelstaägg,ägg m frigående,15.0,st
52656,ica,Räkröra,festköket,räkröra,80.0,g


In [41]:
# For every Coop product, generate the TOP_K most similar ICA products candidates.
# TF-IDF on character n-grams of brand + name, so small spelling differences still match.
# Size is left out on purpose, because same product in another size makes a good hard negative to label.
from sklearn.feature_extraction.text import TfidfVectorizer

TOP_K = 10
ica_c = candidates[candidates["store"] == "ica"].reset_index(drop=True)
coop_c = candidates[candidates["store"] == "coop"].reset_index(drop=True)

def match_text(df):
    return df["brand_norm"] + " " + df["name_norm"]

vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2, sublinear_tf=True)
vectorizer.fit(pd.concat([match_text(ica_c), match_text(coop_c)]))
X_ica = vectorizer.transform(match_text(ica_c))
X_coop = vectorizer.transform(match_text(coop_c))

# Cosine similarity in chunks (the full Coop x ICA matrix would be ~1 billion floats)
rows = []
for start in range(0, X_coop.shape[0], 2000):
    sims = (X_coop[start:start + 2000] @ X_ica.T).toarray()
    top = np.argpartition(-sims, TOP_K, axis=1)[:, :TOP_K]
    for i, idx in enumerate(top):
        for rank, j in enumerate(sorted(idx, key=lambda j: -sims[i, j]), start=1):
            rows.append((start + i, j, rank, sims[i, j]))

pairs = pd.DataFrame(rows, columns=["coop_idx", "ica_idx", "rank", "score"])
print(f"{len(pairs):,} candidate pairs")
pairs.groupby("rank")["score"].median()

268,270 candidate pairs


rank
1     0.809973
2     0.653368
3     0.590563
4     0.553945
5     0.525254
6     0.504108
7     0.484902
8     0.469549
9     0.455710
10    0.444155
Name: score, dtype: float64

In [42]:
# Look at some top-k lists: is the right ICA product usually in there?
def show_candidates(coop_i):
    c = coop_c.loc[coop_i]
    print(f"\nCOOP: {c['name']} | {c['brand']} | {c['size_value']} {c['size_unit']} | {c['price']:.2f} kr")
    for _, r in pairs[pairs["coop_idx"] == coop_i].iterrows():
        i = ica_c.loc[r["ica_idx"]]
        print(f"  {r['rank']:>2.0f}  {r['score']:.2f}  {i['name'][:55]:55} | {i['brand']} "
              f"| {i['size_value']} {i['size_unit']} | {i['price']:.2f} kr")

for coop_i in coop_c.sample(10).index:
    show_candidates(coop_i)


COOP: Salchichón | Zeta | 90.0 g | 42.50 kr
   1  0.54  Salchichón Ibérico Skivad 80g Capa Negra                | CAPA NEGRA | 80.0 g | 49.90 kr
   2  0.36  Feta 150g Zeta                                          | Zeta | 150.0 g | 28.20 kr
   3  0.30  Salladsost 150g Zeta                                    | Zeta | 150.0 g | 24.24 kr
   4  0.29  Chorizo 300g Zeta                                       | Zeta | 300.0 g | 62.20 kr
   5  0.29  Salami Milano 80g Zeta                                  | Zeta | 80.0 g | 36.81 kr
   6  0.28  Salami Napoli 80g Zeta                                  | Zeta | 80.0 g | 36.78 kr
   7  0.28  Ricotta 250g Zeta                                       | Zeta | 250.0 g | 27.88 kr
   8  0.28  Getost 150 g Zeta                                       | Zeta | 150.0 g | 36.13 kr
   9  0.27  Fuet 170g Zeta                                          | Zeta | 170.0 g | 47.05 kr
  10  0.27  Salami Tryffel 60g Zeta                                 | Zeta | 60.0 g | 40

In [43]:
# Save the candidate pairs for the labeling tool.
# label_order is a random order of the Coop products, so labeling goes through a random sample (not just the easy, high-score ones)
out = pairs.assign(
    coop_key=coop_c.loc[pairs["coop_idx"], "product_key"].values,
    ica_key=ica_c.loc[pairs["ica_idx"], "product_key"].values,
)
order = pd.Series(np.random.default_rng(42).permutation(len(coop_c)), index=coop_c["product_key"])
out["label_order"] = out["coop_key"].map(order)

out = out[["label_order", "coop_key", "ica_key", "rank", "score"]].sort_values(["label_order", "rank"])
out.to_parquet("../data/candidate_pairs.parquet", index=False)
products.to_parquet("../data/products_normalized.parquet", index=False)
print(f"saved {len(out):,} pairs for {out['coop_key'].nunique():,} Coop products")

saved 268,270 pairs for 26,827 Coop products


In [ ]:
# Export the first N_EXPORT Coop products (in label_order) with their candidates for the labeling tool (labeler/)
import json

N_EXPORT = 2000  # more than you'll label in one go; rerun with a bigger number if you run out
FIELDS = ["product_key", "name", "brand", "brand_norm", "size_value", "size_unit", "price", "image_url"]

def records(df):
    # product_key -> dict of FIELDS, with NaN turned into None so it becomes null in JSON
    d = df[FIELDS].astype(object)
    return d.where(d.notna(), None).set_index("product_key", drop=False).to_dict("index")

ica_info, coop_info = records(ica_c), records(coop_c)

items = []
for (order, coop_key), group in out[out["label_order"] < N_EXPORT].groupby(["label_order", "coop_key"]):
    items.append({
        "order": int(order),
        "coop": coop_info[coop_key],
        "candidates": [
            {**ica_info[r.ica_key], "rank": int(r.rank), "score": round(float(r.score), 3)}
            for r in group.itertuples()
        ],
    })

export_path = Path("../data/labeling/candidates.json")
export_path.parent.mkdir(parents=True, exist_ok=True)
export_path.write_text(json.dumps({"items": items}, ensure_ascii=False))
print(f"exported {len(items):,} Coop products -> {export_path} ({export_path.stat().st_size / 1e6:.1f} MB)")